<a href="https://colab.research.google.com/github/BlackJackBander/HeadHunter_bot/blob/main/headhunter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import requests
import csv
import time
import re

def clean_html(raw_html):
    """Очищает текст от HTML-тегов, которые HH возвращает в описании."""
    if not raw_html:
        return ""
    clean_text = re.sub(r'<[^>]+>', ' ', raw_html)
    # Убираем лишние пробелы
    return ' '.join(clean_text.split())

def parse_hh_vacancies(search_text, max_pages=1, items_per_page=20):
    base_url = 'https://api.hh.ru/vacancies'

# Измените это на уникальное название вашего приложения
    headers = {
       # 'User-Agent': 'Chrome (contact: your-email@example.com)',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36',
        'Accept': 'application/json'
    }

    all_vacancies_data = []

    print(f"Начинаем поиск по запросу: '{search_text}'...")

    for page in range(max_pages):
        params = {
            'text': search_text,
            #'search_field': ['name', 'description'],
            'page': page,
            'per_page': items_per_page,
            'area': 113 # 113 - Россия. Можно изменить на конкретный город (например, 1 - Москва)
        }

        response = requests.get(base_url, params=params, headers=headers)


        if response.status_code != 200:
            print(f"Ошибка API: {response.status_code}")
            print(f"Детали ошибки от сервера: {response.text}") # <- Добавлена эта строка
            break

        data = response.json()
        items = data.get('items', [])

        if not items:
            print("Больше вакансий не найдено.")
            break

        for item in items:
            vacancy_id = item['id']

            # Запрашиваем детальную информацию по каждой вакансии
            detail_url = f'https://api.hh.ru/vacancies/{vacancy_id}'
            detail_response = requests.get(detail_url, headers=headers)

            if detail_response.status_code == 200:
                detail_data = detail_response.json()

                name = detail_data.get('name', '')
                description = clean_html(detail_data.get('description', ''))

                # Извлекаем ключевые навыки в единую строку
                skills_list = [skill['name'] for skill in detail_data.get('key_skills', [])]
                skills = ", ".join(skills_list)

                area = detail_data.get('area', {}).get('name', '')
                url = detail_data.get('alternate_url', '')

                all_vacancies_data.append([name, description, skills, area, url])

                print(f"Обработана вакансия: {name}")
            else:
                print(f"Не удалось получить детали для вакансии {vacancy_id}")

            # Задержка, чтобы не получить бан от HH за превышение лимита запросов (Rate Limit)
            time.sleep(0.5)

    return all_vacancies_data

def save_to_csv(data, filename='vacancies.csv'):
    # Используем utf-8-sig, чтобы Excel корректно читал кириллицу
    with open(filename, mode='w', encoding='utf-8-sig', newline='') as file:
        writer = csv.writer(file, delimiter=';')
        writer.writerow(['Наименование', 'Описание', 'Ключевые навыки', 'Локация', 'Ссылка'])
        writer.writerows(data)
    print(f"\nДанные успешно сохранены в файл: {filename}")

if __name__ == '__main__':
    # В качестве примера ищем вакансии ИБ
    keyword = "инженер,"

    # Собираем данные (для теста берем 1 страницу, 10 вакансий)
    # Увеличьте max_pages для получения большего объема данных
    vacancies = parse_hh_vacancies(search_text=keyword, max_pages=10, items_per_page=50)

    if vacancies:
        save_to_csv(vacancies, 'hh_vacancies.csv')
    else:
        print("Нет данных для сохранения.")

Начинаем поиск по запросу: 'инженер'...
Обработана вакансия: Инженер
Обработана вакансия: Инженер
Обработана вакансия: Инженер
Обработана вакансия: Инженер
Обработана вакансия: Инженер
Обработана вакансия: Инженер
Обработана вакансия: Инженер
Обработана вакансия: Инженер
Обработана вакансия: Инженер ПТО
Обработана вакансия: Инженер-проектировщик

Данные успешно сохранены в файл: hh_vacancies.csv
